# Data harmonisation

The `unmapped entity` will be converted into `linked_eneties` with the link with other datasets (i.e. secondary, supplementary, and linked_entity).

The harmonisation process:
1. read the unmapped entities, one by one
2. link the unmapped entity one by one, by linking each key property (foreign id); when link to the dataset, check the existing data (using LLM/ollama)
    - if there are existing data (e.g. technology), then use the existing one
    - if no existing data, then create a few on
3. create a linked entity form the unmapped entity, fill with foreign key

### Import Libraries
This cell imports the necessary Python libraries for data manipulation and working with YAML files.

In [1]:
import os
import pandas as pd
import yaml
from pathlib import Path
import csv
import json
import time

In [2]:
import importlib
import harmonise_helpers as hh
importlib.reload(hh)  # pick up edits without restarting the kernel

from harmonise_helpers import (
    MODEL, HARMONISATION_VERSION, ENTITY_CONFIG, SCOPE_CONFIG, ATTR_PATH, ATTR_COLS, LE_PATH, MAPPING_DIR,
    load_registry, append_row, load_attr_registry,
    llm_fill_fields, validate_row,
    resolve_entity, ensure_attr, ensure_scope,
    collect_candidates, generate_audit,
    load_pending_unmapped, mark_unmapped_entities_mapped, mark_all_unmapped_entities_pending,
    backup_derived_data, reset_derived_data,
    start_harmonisation_log, log_harmonisation_event,
)

print("Helpers loaded from harmonise_helpers.py")

Helpers loaded from harmonise_helpers.py


## Run Controls

In [3]:
set_all_unmapped_to_pending = True
create_motel_db_backup = True
reset_motel_db_outputs = True
test_limit = None     # Limit the number of entities to harmonise for testing purposes. Set to None to process all entities.

### Load Controlled Vocabulary Context for LLM
Loads the **Nomenclature** (term definitions for all CV fields) and **Carrier** (valid carrier abbreviations) 
from the source Excel and injects them into the LLM system prompt via `hh.GLOBAL_CV_CONTEXT`.
This gives the LLM full terminology context when filling fields for attributes, scopes, carriers, and other CV types.

In [4]:
_refuel_excel = Path("../1_ingest/ingestion_space/refuel/raw_data/reFuel_TechDatabase_Clean_2026-06-03.xlsx")
_refuel_sheets = pd.read_excel(_refuel_excel, sheet_name=["Nomenclature", "Carrier"])

def _build_cv_context(df_nom, df_car):
    lines = ["The following controlled vocabulary definitions apply to all fields in this database."]

    lines.append("\n--- Nomenclature: term definitions used across all controlled vocabulary fields ---")
    for _, row in df_nom.iterrows():
        term = row.get("Term / Abbreviation")
        defn = row.get("Definition")
        if pd.notna(term) and pd.notna(defn) and str(term).strip():
            lines.append(f"{term}: {defn}")

    lines.append("\n--- Carrier: valid abbreviations for all carrier fields ---")
    for _, row in df_car.iterrows():
        abbr = row.get("Carrier Abbreviation")
        desc = row.get("Carrier Description")
        ctype = row.get("Carrier Type")
        if pd.notna(abbr) and pd.notna(desc) and str(abbr).strip():
            tag = f" ({ctype})" if pd.notna(ctype) else ""
            lines.append(f"{abbr}{tag}: {desc}")

    return "\n".join(lines)

hh.GLOBAL_CV_CONTEXT = _build_cv_context(_refuel_sheets["Nomenclature"], _refuel_sheets["Carrier"])
print(f"CV context loaded ({len(hh.GLOBAL_CV_CONTEXT)} chars, {len(hh.GLOBAL_CV_CONTEXT.splitlines())} lines)")

CV context loaded (13954 chars, 176 lines)


## Load data

### Load All Schema Definitions
This cell defines a function to recursively load all YAML schema files from the `../schema/` directory into a dictionary called `all_schemas`. These schemas are crucial for understanding the structure and validation rules for different data entities.

In [5]:
# Define the base directory for schema files
base_dir = '../schema/'

# Initialize an empty dictionary to hold all loaded schemas
all_schemas = {}

# Function to recursively load YAML files from a given directory
def load_yaml_files(directory):
    for root, dirs, files in os.walk(directory):
        for file in files:
            if file.endswith('.yaml'):
                file_path = os.path.join(root, file)
                with open(file_path, 'r', encoding='utf-8') as f:
                    schema = yaml.safe_load(f)
                    all_schemas[file] = schema

# Call the function to load all YAML files from the schema directory and its subdirectories
load_yaml_files(base_dir)

# Print the number of loaded schemas for verification
print(f"Loaded {len(all_schemas)} schema files.")

Loaded 13 schema files.


### Load Unmapped Entities
This cell loads the unmapped entities from the `unmapped_entities_refuel.yaml` file, which will be used for harmonisation.

In [ ]:
## Load only entities that still need harmonisation.
UNMAPPED_PATH = Path('../motel-db/unmapped_entity/unmapped_entities_refuel.yaml')

if set_all_unmapped_to_pending:
    all_unmapped_entities, _, _ = load_pending_unmapped(UNMAPPED_PATH)
    print("Setting all staged entities to 'to_be_mapped' status...")
    mark_all_unmapped_entities_pending(UNMAPPED_PATH, all_unmapped_entities)

all_unmapped_entities, ue, ue_indices = load_pending_unmapped(UNMAPPED_PATH)

print(f"Staged entities: {len(all_unmapped_entities)}")
print(f"To be mapped: {len(ue)}")
print(f"Already mapped: {len(all_unmapped_entities) - len(ue)}")

Setting all staged entities to 'to_be_mapped' status...


In [ ]:
# Optional test limit. Keep entities and source indices aligned.

if test_limit is not None:
    ue = ue[:test_limit]
    ue_indices = ue_indices[:test_limit]

# Start one persistent log for this harmonisation run.
harmonisation_started = time.perf_counter()
harmonisation_log = start_harmonisation_log(settings={
    "unmapped_path": str(UNMAPPED_PATH.resolve()),
    "schema_path": str(Path(base_dir).resolve()),
    "database_path": str(Path('../motel-db').resolve()),
    "linked_entity_path": str(Path(LE_PATH).resolve()),
    "mapping_path": str(MAPPING_DIR.resolve()),
    "controlled_vocabulary_source": str(_refuel_excel.resolve()),
    "controlled_vocabulary_context_chars": len(hh.GLOBAL_CV_CONTEXT),
    "loaded_schema_count": len(all_schemas),
    "entity_registry_paths": {
        entity_type: str(Path(config["path"]).resolve())
        for entity_type, config in ENTITY_CONFIG.items()
    },
    "llm_model": MODEL,
    "harmonisation_version": HARMONISATION_VERSION,
    "test_limit": test_limit,
    "staged_entities": len(all_unmapped_entities),
    "entities_selected": len(ue),
})
print(f"Harmonisation log: {harmonisation_log}")

### Load Existing Datasets
This cell defines a function to recursively load all CSV data files from the `../motel-db/` directory into a dictionary called `all_csv_data`. These datasets represent existing linked entities and controlled vocabularies that will be used during the harmonisation process.

In [ ]:
# --- Optional: backup then reset all derived data before a fresh run ---
# backup_derived_data() copies current files to motel-db/_backup/<timestamp>/
# reset_derived_data() wipes and recreates each file with schema-correct headers.
# Use the run controls above to decide whether either setup step should run.

_setup_started = time.perf_counter()
setup_actions = []

if create_motel_db_backup:
    backup_derived_data()
    setup_actions.append("backup")
else:
    print("Skipping motel-db backup.")

if reset_motel_db_outputs:
    reset_derived_data()
    setup_actions.append("reset")
else:
    print("Skipping motel-db cleanup/reset.")

log_harmonisation_event(
    harmonisation_log, "setup", "setup_controls_applied",
    duration_seconds=round(time.perf_counter() - _setup_started, 3),
    actions=setup_actions,
    create_motel_db_backup=create_motel_db_backup,
    reset_motel_db_outputs=reset_motel_db_outputs,
)

In [ ]:
# Define the base directory for data files
data_dir = '../motel-db/'

# Initialize an empty dictionary to hold all loaded CSV data
all_csv_data = {}

# Function to recursively load CSV files from a given directory
def load_csv_files(directory):
    for root, dirs, files in os.walk(directory):
        for file in files:
            if file.endswith('.csv'):
                file_path = os.path.join(root, file)
                with open(file_path, mode='r', encoding='utf-8') as f:
                    try:
                        data = pd.read_csv(f)
                    except Exception as e:
                        print(f"Error reading {file_path}: {e}")
                        data = None
                        continue
                    # Use the filename (without path/extension) or full path as key
                    file_name = file.split('.')[0]  # or use file_path for full paSth
                    all_csv_data[file_name] = data

# Load all CSV files from the motel-db directory and its subdirectories
load_csv_files(data_dir)

# Print the number of loaded CSV files for verification
print(f"Loaded {len(all_csv_data)} CSV files.")

# Harmonisation process

### Harmonisation Configuration and Helper Functions
All configuration (entity types, file paths, column definitions) and helper functions
are defined in [harmonise_helpers.py](harmonise_helpers.py) and imported here.

The module provides:
- `ENTITY_CONFIG`, `SCOPE_CONFIG`, `ATTR_PATH`, `LE_PATH`, `MAPPING_DIR` — path and schema config
- `load_registry`, `append_row`, `load_attr_registry` — CSV registry I/O
- `llm_fill_fields`, `validate_row` — schema-guided LLM field filling and validation
- `resolve_entity` — three-stage resolver (exact → LLM → create)
- `ensure_attr`, `ensure_scope` — controlled vocabulary helpers
- `collect_candidates` — deduplication of unmapped entity candidates
- `generate_audit` — per-entity audit report joining all mapping files
- `reset_derived_data` — deletes all derived files (linked entities, attributes, scopes, mappings) while leaving source registries intact

In [ ]:
# all_schemas is already loaded above — make it available to resolve_entity via the module
# (pass it explicitly when calling resolve_entity)
ATTR_SCHEMA = all_schemas.get("attribute.yaml", {})
print(f"Schema keys available: {list(all_schemas.keys())}")

### Step 1: Collect Unique Candidates
This cell iterates through the unmapped entities and extracts unique candidates for technology, process, source, and carrier types. This ensures that each unique name is only resolved once, improving efficiency and consistency.

In [ ]:
# --- Step 1: Collect unique candidates per entity type (dedup so each name is resolved once) ---
_step_started = time.perf_counter()

def collect_candidates(unmapped_entities):
    """
    Extracts unique entity candidates from a list of unmapped entities.

    Iterates over each unmapped entity and collects distinct candidates for
    technology, process, source, and carrier types. Deduplication is done by
    entity name so each unique name is resolved only once in subsequent steps.

    Args:
        unmapped_entities (list[dict]): List of unmapped entity dicts, each containing
            fields like "technology_name", "technology", "sources", and "balancing".

    Returns:
        dict[str, dict]: A mapping from entity type (e.g. "technology", "process",
            "source", "carrier") to a dict of {name: candidate_dict}, where each
            candidate_dict holds the fields needed for registry resolution.
    """
    candidates = {et: {} for et in ENTITY_CONFIG}
    for e in unmapped_entities:
        t = e.get("technology", {})
        tname = e.get("technology_name")
        if tname:
            candidates["technology"].setdefault(tname, {
                "technology_name": tname,
                "technology_type": t.get("technology_type"),
                "technology_category": t.get("technology_category"),
                "technology_description": t.get("technology_description"),
            })
        pname = t.get("process_name")
        if pname:
            candidates["process"].setdefault(pname, {
                "process_name": pname,
                "process_type": t.get("process_type"),
                "process_category": t.get("process_category"),
            })
        for src in e.get("sources", []):
            sname = src.get("source_name")
            if sname:
                candidates["source"].setdefault(sname, {
                    "source_name": sname,
                    "source_description": src.get("source_description"),
                })
        for item in e.get("balancing", {}).get("inputs", []) + e.get("balancing", {}).get("outputs", []):
            cname = item.get("carrier_name")
            if cname:
                candidates["carrier"].setdefault(cname, {"carrier_name": cname})
    return candidates

candidates = collect_candidates(ue)
for et, items in candidates.items():
    print(f"  {et}: {len(items)} unique candidates")

log_harmonisation_event(
    harmonisation_log, "step_1", "candidates_collected",
    duration_seconds=round(time.perf_counter() - _step_started, 3),
    candidate_counts={et: len(items) for et, items in candidates.items()},
)

### Step 2: Resolve Entities via LLM
For each unique candidate collected in Step 1, this cell resolves it against the existing registry using `resolve_entity`. Before matching or creating any registry entry, the LLM first creates a canonical schema-guided name for the candidate. The code does not mechanically rewrite these names. Resolution then proceeds in two stages:

1. **Exact match** — the candidate name is compared case-insensitively against all registry entries; if a match is found the existing ID is reused immediately with no LLM call.
2. **Semantic match (LLM)** — if no exact match exists and the registry is non-empty, the full registry and candidate are sent to the local Ollama model, which returns a JSON verdict indicating whether a semantically equivalent entry already exists and, if so, its ID.

If neither stage finds a match, a new row is created with a freshly generated ID, appended to the CSV, and added to the in-memory registry so subsequent candidates in the same run can match against it.

Step 2 also writes one mapping file per entity type immediately. Each row preserves the original staging name and records the resolved registry name, assigned ID, and resolution status.

In [ ]:
# --- Step 2: Resolve all LLM-backed entities ---
_step_started = time.perf_counter()
registries = {et: load_registry(et) for et in ENTITY_CONFIG}
resolved_ids = {et: {} for et in ENTITY_CONFIG}        # {entity_type: {original_name -> id}}
resolved_names = {et: {} for et in ENTITY_CONFIG}      # {entity_type: {original_name -> registry_name}}
resolution_status = {et: {} for et in ENTITY_CONFIG}   # {entity_type: {original_name -> status}}
MAPPING_DIR.mkdir(parents=True, exist_ok=True)

for entity_type, entity_candidates in candidates.items():
    print(f"\nResolving {entity_type}...")
    registry = registries[entity_type]
    name_field = ENTITY_CONFIG[entity_type]["name_field"]
    counts = {"exact": 0, "llm": 0, "created": 0}
    for name, candidate in entity_candidates.items():
        rid, status = resolve_entity(entity_type, candidate, registry, all_schemas)
        resolved_row = next(row for row in registry if row.get(ENTITY_CONFIG[entity_type]["id_field"]) == rid)
        resolved_ids[entity_type][name] = rid
        resolved_names[entity_type][name] = resolved_row.get(name_field, "")
        resolution_status[entity_type][name] = status
        counts[status] += 1
        resolved_name = resolved_names[entity_type][name]
        if status == "created":
            print(f"  + created: {name!r} -> {resolved_name!r}  [{rid}]")
        log_harmonisation_event(
            harmonisation_log, "step_2", "entity_resolved",
            entity_type=entity_type,
            original_name=name,
            resolved_name=resolved_name,
            resolved_id=rid,
            status=status,
        )

    # Persist original -> resolved name mapping as part of Step 2.
    mapping_path = MAPPING_DIR / f"{entity_type}_map.csv"
    id_field = ENTITY_CONFIG[entity_type]["id_field"]
    with open(mapping_path, "w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(
            f,
            fieldnames=["original_name", name_field, id_field, "status"],
        )
        writer.writeheader()
        for original_name, rid in resolved_ids[entity_type].items():
            writer.writerow({
                "original_name": original_name,
                name_field: resolved_names[entity_type][original_name],
                id_field: rid,
                "status": resolution_status[entity_type][original_name],
            })
    total = sum(counts.values())
    print(f"  total: {total}  |  exact match: {counts['exact']}  |  LLM match: {counts['llm']}  |  newly created: {counts['created']}")
    print(f"  mapping: {mapping_path}")
    log_harmonisation_event(
        harmonisation_log, "step_2", "entity_type_completed",
        entity_type=entity_type,
        counts=counts,
        mapping_path=str(mapping_path.resolve()),
    )

print("\nLLM resolution complete.")
log_harmonisation_event(
    harmonisation_log, "step_2", "completed",
    duration_seconds=round(time.perf_counter() - _step_started, 3),
)

### Step 3: Resolve Controlled Vocabulary (Attributes & Scopes)
This step handles entities whose values come from fixed controlled vocabularies — **attributes** and **scopes** (geographic, temporal, capacity, system boundary). Unlike Steps 1–2, resolution here is fully deterministic: no LLM is used.

- **Attributes** (`ensure_attr`) — first asks the LLM to produce a schema-compliant canonical `attribute_name`, then looks that canonical name up in the in-memory registry; if absent, it generates a new `ATTR_XXXXX` ID and appends a row to `attribute.csv`.
- **Scopes** (`ensure_scope`) — first asks the LLM to create a schema-guided canonical scope token, then checks whether that token already exists in the corresponding CSV; if not, it appends it. The canonical token is used as the foreign key in linked entities.

Both helpers are idempotent: re-running the cell with the same data will not create duplicate entries.

In [ ]:
# --- Step 3: Resolve controlled vocabulary (attributes + scopes) ---
_step_started = time.perf_counter()
ATTR_SCHEMA = all_schemas.get("attribute.yaml", {})

# Repair: rebuild attribute.csv from scratch (delete corrupted file first)
print("Rebuilding attribute.csv from scratch...")
Path(ATTR_PATH).unlink(missing_ok=True)

attr_registry = {}
attr_ids = {}
attr_names = {}
attr_status = {}
scope_ids  = {}
attr_counts  = {"existing": 0, "created": 0}
scope_counts = {"existing": 0, "created": 0}

# Rebuild attributes from all 99 unmapped entities so the registry is complete
with open(r'../motel-db/unmapped_entity/unmapped_entities_refuel.yaml', "r", encoding="utf-8") as f:
    ue_all = yaml.safe_load(f)

for entity in ue_all:
    for attr in entity.get("attributes", []):
        name = attr.get("attribute_name")
        if name and name not in attr_ids:
            aid, canonical_name, status = ensure_attr(name, registry=attr_registry, notes=attr.get("notes", ""), attr_schema=ATTR_SCHEMA)
            attr_ids[name] = aid
            attr_names[name] = canonical_name
            attr_status[name] = status
            attr_counts[status] += 1
            if status == "created":
                print(f"  + attribute: {name!r} -> {canonical_name!r}  [{aid}]")

# Resolve scopes for the working slice only
for entity in ue:
    scope = entity.get("scope", {})
    for scope_type in SCOPE_CONFIG:
        val = scope.get(f"{scope_type}_description")
        key = (scope_type, val)
        if val and key not in scope_ids:
            token, status = ensure_scope(scope_type, val, scope_schema=all_schemas.get(f"{scope_type}.yaml", {}))
            scope_ids[key] = token
            if status:
                scope_counts[status] += 1
                if status == "created":
                    print(f"  + {scope_type}: {val!r} -> {token!r}")

print(f"Attributes   — total: {sum(attr_counts.values())}  |  existing: {attr_counts['existing']}  |  created: {attr_counts['created']}")
print(f"Scope tokens — total: {sum(scope_counts.values())}  |  existing: {scope_counts['existing']}  |  created: {scope_counts['created']}")
log_harmonisation_event(
    harmonisation_log, "step_3", "controlled_vocabulary_resolved",
    duration_seconds=round(time.perf_counter() - _step_started, 3),
    attribute_counts=attr_counts,
    scope_counts=scope_counts,
    attribute_path=str(Path(ATTR_PATH).resolve()),
)

### Step 4: Build Linked Entity Records and Save
With all foreign keys resolved (technology, process, source, carrier, attributes, scopes), this step assembles the final `linked_entity` records and writes them to `linked_entity/linked_entity.yaml`.

YAML is used instead of CSV because the linked entity schema is deeply nested — sources, balancing flows, and attribute values are arrays of objects that do not flatten cleanly into tabular columns.

For each unmapped entity:
- **Foreign keys** for technology and process are looked up from `resolved_ids` (Step 2).
- **Scope tokens** are looked up from `scope_ids` (Step 3).
- **Balancing flows** (inputs/outputs) are structured as `{carrier_id, share, unit}` objects.
- **Sources** are structured as `{source_id, linked_attribute_ids}` objects.
- **Attribute values** are structured as `{attribute_id, attribute_name, value, time_index}` objects.

The output file is overwritten on every run.

In [ ]:
# --- Step 4: Build linked_entity records and save as YAML ---
_step_started = time.perf_counter()
from datetime import date

linked_entities = []
today = str(date.today())

# Preserve previously mapped records when processing a later ingestion batch.
le_path = Path(LE_PATH)
if le_path.exists():
    with open(le_path, "r", encoding="utf-8") as f:
        existing_linked_entities = yaml.safe_load(f) or []
else:
    existing_linked_entities = []

existing_numbers = [
    int(le["linked_entity_id"].split("_")[-1])
    for le in existing_linked_entities
    if str(le.get("linked_entity_id", "")).startswith("LE_")
]
next_linked_number = max(existing_numbers, default=0) + 1

for i, entity in enumerate(ue):
    t     = entity.get("technology", {})
    scope = entity.get("scope", {})

    linked_entities.append({
        "linked_entity_id": f"LE_{next_linked_number+i:05d}",
        "tech_id":     resolved_ids["technology"].get(entity.get("technology_name"), ""),
        "process_id":  resolved_ids["process"].get(t.get("process_name"), ""),
        "scope": {
            "geographic_scope": scope_ids.get(("geographic_scope", scope.get("geographic_scope_description")), ""),
            "temporal_scope":   scope_ids.get(("temporal_scope",   scope.get("temporal_scope_description")), ""),
            "capacity_scope":   scope_ids.get(("capacity_scope",   scope.get("capacity_scope_description")), ""),
            "system_boundary":  scope_ids.get(("system_boundary",  scope.get("system_boundary_description")), ""),
        },
        "balancing": {
            "inputs":  [
                {"carrier_id": resolved_ids["carrier"].get(x["carrier_name"], ""), "share": x.get("share"), "unit": x.get("unit")}
                for x in entity.get("balancing", {}).get("inputs", []) if x.get("carrier_name")
            ],
            "outputs": [
                {"carrier_id": resolved_ids["carrier"].get(x["carrier_name"], ""), "share": x.get("share"), "unit": x.get("unit")}
                for x in entity.get("balancing", {}).get("outputs", []) if x.get("carrier_name")
            ],
        },
        "sources": [
            {
                "source_id": resolved_ids["source"].get(s["source_name"], ""),
                "linked_attribute_ids": [
                    attr_ids.get(a) or f"[unregistered: {a}]"
                    for a in s.get("linked_attribute", [])
                ],
            }
            for s in entity.get("sources", []) if s.get("source_name")
        ],
        "values": [
            {
                "attribute_id":   attr_ids.get(a["attribute_name"], ""),
                "attribute_name": attr_names.get(a["attribute_name"], a["attribute_name"]),
                "value":          a.get("value"),
                "time_index":     a.get("time_index"),
            }
            for a in entity.get("attributes", []) if a.get("attribute_name")
        ],
        "date_created": today,
    })

all_linked_entities = existing_linked_entities + linked_entities
le_path.parent.mkdir(parents=True, exist_ok=True)
temporary_le_path = le_path.with_suffix(le_path.suffix + ".tmp")
with open(temporary_le_path, "w", encoding="utf-8") as f:
    yaml.safe_dump(all_linked_entities, f, allow_unicode=True, sort_keys=False)
temporary_le_path.replace(le_path)

# Change status only after the linked-entity output was saved successfully.
mark_unmapped_entities_mapped(
    UNMAPPED_PATH,
    all_unmapped_entities,
    ue_indices,
    linked_entities,
    today,
)

print(f"Saved {len(linked_entities)} new linked entities -> {LE_PATH}")
print(f"Updated mapping status to mapped for {len(linked_entities)} staged entities")
for le in linked_entities:
    print(f"  {le['linked_entity_id']}  tech={le['tech_id']}  process={le['process_id']}  scope={le['scope']}")

log_harmonisation_event(
    harmonisation_log, "step_4", "linked_entities_saved",
    duration_seconds=round(time.perf_counter() - _step_started, 3),
    created_count=len(linked_entities),
    preserved_count=len(existing_linked_entities),
    output_path=str(le_path.resolve()),
    linked_entity_ids=[le["linked_entity_id"] for le in linked_entities],
)

### Step 5: Save Mapping Files and Audit
After harmonisation, two types of mapping files are written to `motel-db/mapping/` to record how every input name was resolved. These files are useful for auditing, re-running, and tracing data lineage.

**Provenance map** (`unmapped_to_linked.csv`) — one row per unmapped entity, linking its original index and technology name to the resulting `linked_entity_id` and all resolved foreign keys (tech, process, scope, sources). This is the primary traceability record: given any linked entity, you can trace it back to the exact unmapped input.

**Entity lookup maps** for technology, process, source, and carrier are created in Step 2. Each preserves the original staging name, resolved registry name, assigned ID, and resolution status. Step 5 adds the attribute and scope maps without overwriting the Step 2 maps.

**Per-entity audit report** — after the mapping files are saved, a per-entity audit joins all resolved mappings and prints a structured summary for each entity: assigned IDs, resolution status for technology, process, sources, carriers, and scope, plus any `unresolved_attributes`. Use this to spot-check the harmonisation output before committing results.

In [ ]:
# --- Step 5: Save mapping files ---
_step_started = time.perf_counter()
MAPPING_DIR.mkdir(parents=True, exist_ok=True)

# --- Provenance map: one row per unmapped entity -> linked entity ---
MAP_A_PATH = MAPPING_DIR / "unmapped_to_linked.csv"
MAP_A_COLS = [
    "unmapped_index", "technology_name",
    "linked_entity_id", "tech_id", "process_id",
    "geographic_scope", "temporal_scope", "capacity_scope", "system_boundary",
    "source_ids", "date_mapped",
]

with open(MAP_A_PATH, "w", newline="", encoding="utf-8") as f:
    writer = csv.DictWriter(f, fieldnames=MAP_A_COLS)
    writer.writeheader()
    for source_index, entity, le in zip(ue_indices, ue, linked_entities):
        source_ids = [s["source_id"] for s in le.get("sources", [])]
        writer.writerow({
            "unmapped_index":   source_index,
            "technology_name":  entity.get("technology_name", ""),
            "linked_entity_id": le["linked_entity_id"],
            "tech_id":          le["tech_id"],
            "process_id":       le["process_id"],
            "geographic_scope": le["scope"].get("geographic_scope", ""),
            "temporal_scope":   le["scope"].get("temporal_scope", ""),
            "capacity_scope":   le["scope"].get("capacity_scope", ""),
            "system_boundary":  le["scope"].get("system_boundary", ""),
            "source_ids":       json.dumps(source_ids),
            "date_mapped":      today,
        })

print(f"Provenance map: saved {len(linked_entities)} rows -> {MAP_A_PATH}")

# Technology, process, source, and carrier maps were written in Step 2.
# Attribute lookup map
attr_path = MAPPING_DIR / "attribute_map.csv"
with open(attr_path, "w", newline="", encoding="utf-8") as f:
    writer = csv.DictWriter(f, fieldnames=["original_name", "attribute_name", "attribute_id", "status"])
    writer.writeheader()
    for name, aid in attr_ids.items():
        writer.writerow({
            "original_name": name,
            "attribute_name": attr_names.get(name, name),
            "attribute_id": aid,
            "status": attr_status.get(name, "created"),
        })
print(f"Entity lookup map: attribute_map.csv  ({len(attr_ids)} rows)")

# Scope lookup maps
for scope_type in SCOPE_CONFIG:
    path = MAPPING_DIR / f"{scope_type}_map.csv"
    scope_entries = {val: token for (st, val), token in scope_ids.items() if st == scope_type}
    with open(path, "w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=["original_value", "scope_token", "status"])
        writer.writeheader()
        for val, token in scope_entries.items():
            writer.writerow({"original_value": val, "scope_token": token, "status": "created"})
    print(f"Entity lookup map: {scope_type}_map.csv  ({len(scope_entries)} rows)")

log_harmonisation_event(
    harmonisation_log, "step_5", "mapping_files_saved",
    duration_seconds=round(time.perf_counter() - _step_started, 3),
    provenance_rows=len(linked_entities),
    provenance_path=str(MAP_A_PATH.resolve()),
    attribute_map_path=str(attr_path.resolve()),
    entity_mapping_paths={
        entity_type: str((MAPPING_DIR / f"{entity_type}_map.csv").resolve())
        for entity_type in ENTITY_CONFIG
    },
    scope_mapping_paths={
        scope_type: str((MAPPING_DIR / f"{scope_type}_map.csv").resolve())
        for scope_type in SCOPE_CONFIG
    },
)

In [ ]:
# --- Per-entity audit report: joins all resolved mappings and prints a structured summary for each entity ---
audit_indices = [0, 1, 2]
audit_results = generate_audit(ue, attr_ids, indices=audit_indices, source_indices=ue_indices)

print("=== Per-Entity Audit Report ===")
print(f"Auditing {len(audit_results)} of {len(ue)} entities.")
print("Each entry shows the linked entity ID and how every sub-entity")
print("(technology, process, sources, carriers, scope) was resolved.\n")
for entry in audit_results:
    print(json.dumps(entry, indent=2))

log_harmonisation_event(
    harmonisation_log, "audit", "completed",
    audited_indices=audit_indices,
    audited_count=len(audit_results),
)
log_harmonisation_event(
    harmonisation_log, "run", "completed",
    total_duration_seconds=round(time.perf_counter() - harmonisation_started, 3),
    mapped_entities=len(linked_entities),
)
print(f"\nHarmonisation complete. Log saved to: {harmonisation_log}")